<a href="https://colab.research.google.com/github/martirossi/AppliedML2026_mr/blob/main/classification_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# ============================================
# XGBoost + SHAP Feature Selection + Final Test Predictions
# ============================================

import numpy as np
import pandas as pd
import xgboost as xgb
import shap
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss

# ------------------------------------------------
# 1. Load training data
# ------------------------------------------------
train_path = "/content/drive/MyDrive/Colab Notebooks/AppML_InitialProject_train.csv"
data = pd.read_csv(train_path)

target_col = "p_Truth_isElectron"
y = data[target_col].values

# Detect ID column if present
id_col = None
for col in data.columns:
    if col.lower() in ["eventid", "id"]:
        id_col = col

feature_cols = [c for c in data.columns if c not in [target_col, id_col]]

X_full = data[feature_cols].values




In [15]:
# ------------------------------------------------
# 2. Train a temporary XGBoost model on ALL features
# ------------------------------------------------
temp_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    tree_method="hist"
)

temp_model.fit(X_full, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
# ------------------------------------------------
# 3. Compute SHAP values and rank features (truth excluded)
# ------------------------------------------------

# Ricrea la lista delle feature (in caso il runtime sia stato riavviato)
feature_cols = [c for c in data.columns if c not in [target_col, id_col]]

# Escludi TUTTE le variabili di verità
allowed_features = [c for c in feature_cols if not c.startswith("p_Truth_")]

# Costruisci X solo con feature valide
X_allowed = data[allowed_features].values

# Re-train temporary XGBoost model on ONLY allowed features for SHAP consistency
temp_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    tree_method="hist"
)
temp_model.fit(X_allowed, y) # Train with X_allowed

# SHAP sul modello temporaneo
explainer = shap.TreeExplainer(temp_model)
shap_values = explainer.shap_values(X_allowed)

# Mean absolute SHAP value per feature
shap_importance = np.abs(shap_values).mean(axis=0)

# Ranking
shap_ranking = pd.DataFrame({
    "Feature": allowed_features,
    "SHAP_Importance": shap_importance
}).sort_values(by="SHAP_Importance", ascending=False)

# Select top 15 features
top_15_features = shap_ranking["Feature"].head(15).tolist()

print("Selected 15 features using SHAP:")
for f in top_15_features:
    print(f)


In [ ]:
# ------------------------------------------------
# 4. Prepare data with selected features
# ------------------------------------------------
X = data[top_15_features].values

In [ ]:
# ------------------------------------------------
# 5. 5-fold CV with XGBoost
# ------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(data))

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "eta": 0.03,
    "max_depth": 6,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "lambda": 1.0,
    "alpha": 0.0
}

fold = 1
for train_idx, valid_idx in kf.split(X, y):
    print(f"\n===== Fold {fold} =====")

    X_train, X_valid = X[train_idx], X[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=500,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=50,
        verbose_eval=50
    )

    best_iter = model.best_iteration
    print("Best iteration:", best_iter)

    oof_preds[valid_idx] = model.predict(xgb.DMatrix(X_valid), iteration_range=(0, best_iter))

    fold += 1

In [ ]:
# ------------------------------------------------
# 6. Overall OOF LogLoss
# ------------------------------------------------
overall_logloss = log_loss(y, oof_preds)
print("\nOverall OOF LogLoss (5-fold CV):", overall_logloss)

In [ ]:
# ------------------------------------------------
# 7. Train final model on ALL data
# ------------------------------------------------
final_dtrain = xgb.DMatrix(X, label=y)
final_model = xgb.train(
    params,
    final_dtrain,
    num_boost_round=int(np.mean([model.best_iteration for _ in range(5)]))
)


In [ ]:
# ------------------------------------------------
# 8. Load test data and predict
# ------------------------------------------------
test_path = "/content/drive/MyDrive/Colab Notebooks/AppML_InitialProject_test_classification.csv"
test_data = pd.read_csv(test_path)

# Detect ID column
test_id_col = None
for col in test_data.columns:
    if col.lower() in ["eventid", "id"]:
        test_id_col = col
        break

if test_id_col is not None:
    test_ids = test_data[test_id_col].values
else:
    test_ids = np.arange(len(test_data))

# Filter out 'p_Truth_Energy' from top_15_features if it exists, as it's a target-related feature
predictive_features = [f for f in top_15_features if f != 'p_Truth_Energy']
X_test = test_data[predictive_features].values
test_preds = final_model.predict(xgb.DMatrix(X_test))

In [ ]:
# ------------------------------------------------
# 9. Save required files (NO HEADERS)
# ------------------------------------------------

# File 1: predictions (NO column names)
submission_df = pd.DataFrame({
    "ID": test_ids,
    "Predicted_Probability": test_preds
})
submission_df.to_csv("Classification_MartinaRossi_XGBoost.csv", index=False, header=False)

# File 2: variable list (NO column name, NO truth vars)
clean_features = [f for f in top_15_features if not f.startswith("p_Truth_")]

varlist_df = pd.DataFrame(clean_features)
varlist_df.to_csv("Classification_MartinaRossi_XGBoost_VariableList.csv", index=False, header=False)

print("\nSaved files WITHOUT HEADERS:")
print(" - Classification_MartinaRossi_XGBoost.csv")
print(" - Classification_MartinaRossi_XGBoost_VariableList.csv")
